# Indiviudal Analysis


In [1]:
import pandas as pd
import datetime 
import numpy as np
import statsmodels.formula.api as smf


In [2]:
daily = pd.read_csv('../data/daily.csv').drop(columns=["Unnamed: 0", 'Unnamed: 0.1'])
birth_year = pd.to_numeric(daily['birth_date'].str[-4:], errors='coerce')
death_year = pd.to_numeric(daily['death_month'].astype(str).str.split('.').str[-1], errors='coerce')
daily['age'] = death_year - birth_year
daily['Date'] = pd.to_datetime(daily['death_month'].astype(str), format='%m.%Y', errors ='coerce').dt.strftime('%Y-%m-%d')


In [3]:
contract = ['автомобильные', 'артиллерия', 'ВДВ', 'военмед', 'военные пилоты',
       'войска связи', 'войсковая ПВО', 'инженерные войска', 'МВД','морпехи', 'моряки',
       'мотострелковые войска', 'наземные авиаслужбы', 'нацгвардия',
       'РХБЗ', 'спецназ', 'танковые войска', 'ФСБ', 
       'другие войска', 'ЖД', 'военная полиция', 'СК', 'ФСО']

daily["total"] = 1
daily["drafted"] = daily["pmc"] = daily["volunteers"] = daily["prisoners"] = daily['contract'] = 0
daily.loc[daily['branch'] == 'добровольцы', 'volunteers'] = 1
daily.loc[daily['branch'] == 'мобилизованные', 'drafted'] = 1
daily.loc[daily['branch'] == 'ЧВК', 'pmc'] = 1
daily.loc[daily['branch'] == 'заключенные', 'prisoners'] = 1
daily.loc[daily['branch'] ==  'нет данных', 'unknown'] = 1
daily.loc[daily['branch'].isin(contract), 'contract'] = 1


def create_branch_category(row):
    if row['volunteers'] == 1:
        return 'volunteers'
    elif row['drafted'] == 1:
        return 'drafted'
    elif row['pmc'] == 1:
        return 'pmc'
    elif row['prisoners'] == 1:
        return 'prisoners'
    elif row['unknown'] == 1:
        return 'unknown'
    elif row['contract'] == 1:
        return 'contract'
    else:
        return 'other'

daily['branch_category'] = daily.apply(create_branch_category, axis=1)
daily[["drafted", "total", "contract",
     "prisoners", "volunteers", "pmc", 'unknown']] = daily[["drafted", "total", "contract",
                                               "prisoners", "volunteers", "pmc", 'unknown']].fillna(0)


In [4]:
df = daily[daily["branch_category"] == 'drafted']
df['Date'] = pd.to_datetime(df['Date'])
reference_date = pd.to_datetime('2022-09-01')
df = df[df['Date'] >= reference_date ]
df = df[['region', 'Date', 'slavic', 'age']]

df['months_from_draft'] = ((pd.to_datetime(df['Date']).dt.year - reference_date.year) * 12 + 
                               (pd.to_datetime(df['Date']).dt.month - reference_date.month))
df['months_from_draft'] = df['months_from_draft'].astype('Int64')
df.rename(columns = {'region':'Region'}, inplace = True)
df

C:\Users\Albert\AppData\Local\Temp\ipykernel_14472\1738679857.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date'] = pd.to_datetime(df['Date'])


,Region,Date,slavic,age,months_from_draft
1073,Алтайский край,2024-08-01,1,53.0,23
1074,Алтайский край,2024-08-01,1,53.0,23
1075,Алтайский край,2024-08-01,1,53.0,23
1076,Алтайский край,2024-08-01,1,53.0,23
1077,Алтайский край,2024-08-01,1,53.0,23
...,...,...,...,...,...
118362,Неизвестно,2023-01-01,1,NaN,4
119023,Иностранцы,2023-01-01,0,27.0,4
119024,Иностранцы,2023-01-01,0,27.0,4
119025,Иностранцы,2023-01-01,0,27.0,4


In [5]:
df_cs = pd.read_csv('../data/intermediate/annual.csv')
df_cs = df_cs.drop(columns=["Unnamed: 0"])
df_cs = df_cs[df_cs['year'] == 2022]
df_cs = df_cs[['Region', "% Russians", 'population', 'urban_share', 'median_income', 'share_poverty']]
df_cs['urban_share'] = df_cs['urban_share']*100
df_cs['log_pop'] = np.log(df_cs['population'])
df_cs['log_inc'] = np.log(df_cs['median_income'])

In [9]:
df_gov = pd.read_csv('../data/gov.csv')
today = pd.to_datetime('today')
df_gov['start'] = pd.to_datetime(df_gov['start'], errors='coerce')
df_gov['end'] = pd.to_datetime(df_gov['end'], errors='coerce')
df_gov['end'] = df_gov['end'].fillna(today)
df_gov = df_gov[(df_gov['end'] > '2022-12-01') & (df_gov['start'] < '2022-09-01')]
df_gov['yio'] = (reference_date.year - pd.to_datetime(df_gov['start']).dt.year)
df_gov

,Unnamed: 0.1,Unnamed: 0,region_type,region,position,governor,start,end,birth_year,insider,approved,nat_rep,Region,turnout21,ur21,elections22,elections23,elections24,yio
0,0,0,Republics,Adygeya,"Presidents (from 1 May 2011, Heads of the Repu...",Murat Kumpilov,2017-01-12,2025-08-27 17:38:18.474543,1973,1,1,1,Республика Адыгея,68.21,66.45,0,0,0,5
1,1,1,Republics,Altay,"Heads of the Republic, Chairmen of the Government",Oleg Khorokhordin,2019-03-20,2024-06-04 00:00:00.000000,1972,1,1,1,Республика Алтай,46.19,38.50,0,0,1,3
3,3,3,Republics,Bashkortostan,"Presidents (from 1 Jan 2015, Heads of the Repu...",Rady Khabirov,2018-10-11,2025-08-27 17:38:18.474543,1964,1,1,1,Республика Башкортостан,72.79,66.61,0,0,1,4
4,4,4,Republics,Buryatia,"Presidents (from 12 May 2012, Heads of the Rep...",Aleksey Tsydenov,2017-02-07,2025-08-27 17:38:18.474543,1976,1,1,1,Республика Бурятия,44.97,42.63,0,0,0,5
5,5,5,Republics,Chechnya,"Presidents (from 1 May 2011, Heads of the Repu...",Ramzan Kadyrov,2007-02-15,2025-08-27 17:38:18.474543,1976,1,1,1,Чеченская Республика,94.42,96.13,0,0,0,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,103,103,Autonomous oblast (province),Yevreyskaya,Heads of the administration,Rostislav Goldshteyn,2019-12-12,2024-11-05 00:00:00.000000,1969,0,1,1,Еврейская АО,63.13,56.39,0,0,0,3
105,105,105,Autonomous okruga (districts),Chukotka,Heads of the administration,Roman Kopin,2008-07-03,2023-03-15 00:00:00.000000,1974,1,1,1,Чукотский АО,61.29,46.71,0,1,0,14
107,107,107,Autonomous okruga (districts),Khanty-Mansi,"Head of the administration (from Oct 1996, gov...",Natalya Komarova (f),2010-03-01,2024-05-30 00:00:00.000000,1955,0,1,1,Ханты-Мансийский АО,46.62,42.30,0,0,0,12
109,109,109,Autonomous okruga (districts),Nenets,Heads of the administration,Yury Bezdudny,2020-04-02,2025-03-18 00:00:00.000000,1969,1,1,1,Ненецкий АО,42.61,29.06,0,0,0,2


In [6]:
df_gov = pd.read_csv('../data/gov.csv')
today = pd.to_datetime('today')
df_gov['start'] = pd.to_datetime(df_gov['start'], errors='coerce')
df_gov['end'] = pd.to_datetime(df_gov['end'], errors='coerce')
df_gov['end'] = df_gov['end'].fillna(today)
df_gov = df_gov[(df_gov['end'] > '2022-12-01') & (df_gov['start'] < '2022-09-01')]
df_gov['yio'] = (reference_date.year - pd.to_datetime(df_gov['start']).dt.year)
df_gov = df_gov[['region', 'Region','insider', 'nat_rep', 'yio']]
reg_name_map = {'Архангельская область без АО':'Архангельская область',
 'Еврейская АО':'Еврейская автономная область',
 'Кабардино-Балкария':'Кабардино-Балкарская Республика',
 'Карачаево-Черкесия': 'Республика Карачаево-Черкесия',
 'Ненецкий АО':'Ненецкий автономный округ',
 'Якутия':'Республика Саха (Якутия)',
 'Северная Осетия':'Республика Северная Осетия-Алания',
 'Тюменская область без АО':'Тюменская область',
 'Ханты-Мансийский АО':'Ханты-Мансийский автономный округ - Югра',
 'Ямало-Hенецкий АО':'Ямало-Ненецкий автономный округ',
 'Чукотский АО':'Чукотский автономный округ'
}


df_gov['Region'] = df_gov['Region'].replace(reg_name_map)
df_gov

,region,Region,insider,nat_rep,yio
0,Adygeya,Республика Адыгея,1,1,5
1,Altay,Республика Алтай,1,1,3
3,Bashkortostan,Республика Башкортостан,1,1,4
4,Buryatia,Республика Бурятия,1,1,5
5,Chechnya,Чеченская Республика,1,1,15
...,...,...,...,...,...
103,Yevreyskaya,Еврейская автономная область,0,1,3
105,Chukotka,Чукотский автономный округ,1,1,14
107,Khanty-Mansi,Ханты-Мансийский автономный округ - Югра,0,1,12
109,Nenets,Ненецкий автономный округ,1,1,2


In [7]:
reg_name_map = {'Архангельская область без АО':'Архангельская область',
 'Еврейская АО':'Еврейская автономная область',
 'Кабардино-Балкария':'Кабардино-Балкарская Республика',
 'Карачаево-Черкесия': 'Республика Карачаево-Черкесия',
 'Ненецкий АО':'Ненецкий автономный округ',
 'Якутия':'Республика Саха (Якутия)',
 'Северная Осетия':'Республика Северная Осетия-Алания',
 'Тюменская область без АО':'Тюменская область',
 'Ханты-Мансийский АО':'Ханты-Мансийский автономный округ - Югра',
 'Ямало-Hенецкий АО':'Ямало-Ненецкий автономный округ'

}

df_cs['Region'] = df_cs['Region'].replace(reg_name_map)
df = df.merge(df_cs, on = 'Region', how = 'left')
df = df.merge(df_gov, on = 'Region', how = 'left')

In [8]:
model = smf.ols('months_from_draft ~ age + slavic + log_inc + nat_rep + insider',
                 data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:      months_from_draft   R-squared:                       0.041
Model:                            OLS   Adj. R-squared:                  0.040
Method:                 Least Squares   F-statistic:                     95.55
Date:                 Ср, 27 авг 2025   Prob (F-statistic):           6.48e-99
Time:                        17:36:54   Log-Likelihood:                -38699.
No. Observations:               11317   AIC:                         7.741e+04
Df Residuals:                   11311   BIC:                         7.745e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     25.0525      2.998      8.358      0.0